# Post\-Training Quantization

## Setup

### Import modules

In [39]:
import numpy as np
import onnx
import onnxruntime as ort
import torch
from onnxruntime.quantization import (
    CalibrationDataReader,
    CalibrationMethod,
    QuantFormat,
    QuantType,
    quantize_dynamic,
    quantize_static,
)
from onnxruntime.quantization.matmul_nbits_quantizer import (
    DefaultWeightOnlyQuantConfig,
    MatMulNBitsQuantizer,
)
from tokenizers import Tokenizer
from torch.utils.data import Dataset
from transformers import AutoTokenizer

/usr/local/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Evaluation Dataset

In [41]:
MAX_LENGTH = 64
MODEL_NAME = "huawei-noah/TinyBERT_General_4L_312D"
LABELS = [
    "enable_cloud",
    "disable_cloud",
    "increase_interval",
    "decrease_interval",
    "unknown",
]
LABEL_TO_ID = {label: index for index, label in enumerate(LABELS)}

class IntentDataset(Dataset):
    def __init__(
        self,
        path,
        tokenizer,
        label2id,
        max_length,
    ):
        texts = []
        labels = []
        with open(path, "r") as fp:
            for line in fp:
                row = json.loads(line)
                texts.append(row["text"])
                labels.append(row["label"])

        tokenized_data = tokenizer(
            texts,
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_tensors="pt",
        )
        self.input_ids = tokenized_data["input_ids"]
        self.attention_mask = tokenized_data["attention_mask"]
        self.token_type_ids = tokenized_data["token_type_ids"]
        self.labels = torch.tensor(
            [label2id[label] for label in labels], dtype=torch.long
        )

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, index: int) -> dict[str, torch.Tensor]:
        return {
            "input_ids": self.input_ids[index],
            "attention_mask": self.attention_mask[index],
            "token_type_ids": self.token_type_ids[index],
            "labels": self.labels[index],
        }

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
test_data = IntentDataset("test.jsonl", tokenizer, LABEL_TO_ID, MAX_LENGTH) 

### Evaluation function

In [73]:
def test_accuracy(model_path, test_data):
    session = ort.InferenceSession(str(model_path), providers=["CPUExecutionProvider"])
    correct = 0
    seen = 0
    for sample in test_data:
        input_ids = sample["input_ids"].numpy()[None, :]
        attention_mask = sample["attention_mask"].numpy()[None, :]
        label = sample["labels"]
        logits = session.run(["logits"], {"input_ids": input_ids, "attention_mask": attention_mask})[0]
        predicted = logits[0].argmax()
        correct += int(predicted == label)
        seen += 1
    return correct / seen

In [75]:
fp32_accuracy = test_accuracy('./results/intent_classifier.onnx', test_data)
print(f"FP32 baseline: accuracy={fp32_accuracy*100:.2f}%, ")

FP32 baseline: accuracy=98.00%, 


## Weight\-only Quantization

In [ ]:
def quantize_weight_only(source, destination):
    config = DefaultWeightOnlyQuantConfig(
        bits=4,
        block_size=128,
        is_symmetric=False,
        accuracy_level=4,
        quant_format=QuantFormat.QOperator,
        op_types_to_quantize=("MatMul", "Gather"),
        quant_axes=(("MatMul", 0), ("Gather", 1)),
    )
    quantizer = MatMulNBitsQuantizer(str(source), algo_config=config)
    quantizer.process()
    onnx.save_model(quantizer.model.model, destination, save_as_external_data=False)

## Weights \+ Activations Quantization

### Create a Calibration Set

In [ ]:
class DataReader(CalibrationDataReader):
    def __init__(self, dataset):
        self.dataset = dataset
        self.enum_data = None

        self.datasize = len(self.dataset)

    def get_next(self):
        if self.enum_data is None:
            self.enum_data = iter(self.dataset)

        x = next(self.enum_data, None)

        if x is None:
            return None

        x = x['x']
        x = x.numpy()
        x = np.expand_dims(x, 0)
        x = ort_frontend.run(None, {'input': x})[0]
        x = {'input': x}

        return x

    def rewind(self):
        self.enum_data = None

### Dynamic Quantization

In [ ]:
def quantize_dynamic_int8(source, destination, reduce_range):
    quantize_dynamic(
        source,
        destination,
        per_channel=True,
        reduce_range=reduce_range,
        weight_type=QuantType.QInt8,
        op_types_to_quantize=["MatMul", "Gemm"],
    )

### Static Quantization

In [ ]:
def quantize_static_int8(
    source,
    destination,
    calibration_reader
) -> None:
    quantize_static(
        str(source),
        str(destination),
        calibration_reader,
        quant_format=QuantFormat.QDQ,
        op_types_to_quantize=["MatMul", "Gemm"],
        per_channel=True,
        activation_type=QuantType.QInt8,
        weight_type=QuantType.QInt8,
        calibrate_method="minmax",
    )

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=ef1a6b0f-e832-4c16-b5e1-88d9ff585390' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>